# 🎬 Classification de Critiques de Films (IMDB) — Réseau Dense

## Présentation du projet

Dans ce notebook, nous allons construire un modèle de **classification binaire de texte** pour déterminer si une critique de film est **positive** ou **négative**, en utilisant le dataset IMDB (50 000 critiques).

### 🎯 Ce que vous allez apprendre :
- Prétraiter du texte pour un réseau de neurones (one-hot encoding)
- Construire un réseau **feedforward** (Dense) pour la classification binaire
- Évaluer un modèle avec des ensembles de validation et de test
- Détecter l'**overfitting** grâce aux courbes d'entraînement

### 💡 Idée clé
Chaque critique est déjà encodée comme une **séquence d'entiers** (indices des mots les plus fréquents). Comme un réseau Dense attend des **vecteurs de taille fixe**, on transforme chaque séquence en un vecteur binaire de 10 000 dimensions (one-hot encoding multi-label) : `1` si le mot est présent dans la critique, `0` sinon.

---
## 📦 PARTIE 1 — Prétraitement des données

**Objectif :** Charger le dataset IMDB, le transformer en vecteurs binaires utilisables par un réseau Dense, et créer les splits train/validation/test.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_theme(style='darkgrid')
%matplotlib inline

print(f"✅ TensorFlow {tf.__version__} importé")
print(f"🖥️  GPU disponible : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Chargement du dataset IMDB ───────────────────────────────────────────────
# On garde uniquement les 10 000 mots les plus fréquents du vocabulaire
# → réduit la dimensionnalité tout en gardant l'essentiel du signal

num_words = 10_000

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(num_words=num_words)

print(f"✅ Dataset IMDB chargé !")
print(f"   Train : {len(train_data)} critiques")
print(f"   Test  : {len(test_data)} critiques")

In [ ]:
# ── Exploration de la structure des données ──────────────────────────────────
print("🔍 Exemple de critique (encodée en entiers) :")
print(train_data[0][:20], "...")
print(f"\n   Longueur de cette critique : {len(train_data[0])} mots")
print(f"   Label associé : {train_labels[0]} ({'positif' if train_labels[0] == 1 else 'négatif'})")

print(f"\n📊 Distribution des labels (train) :")
print(f"   Négatifs (0) : {(train_labels == 0).sum()} ({(train_labels == 0).mean()*100:.1f}%)")
print(f"   Positifs (1) : {(train_labels == 1).sum()} ({(train_labels == 1).mean()*100:.1f}%)")

# Vérifier qu'aucun indice ne dépasse num_words
max_index = max(max(seq) for seq in train_data)
print(f"\n✅ Indice maximum dans les séquences : {max_index} (doit être < {num_words})")

In [ ]:
# ── Décodage d'une critique en texte lisible (à titre illustratif) ──────────
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(encoded_review):
    """Convertit une séquence d'entiers en texte lisible."""
    # Les indices 0, 1, 2 sont réservés (padding, début de séquence, inconnu)
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

print("📝 Exemple de critique décodée :\n")
print(decode_review(train_data[0])[:500], "...")
print(f"\n   → Label : {'😊 Positif' if train_labels[0] == 1 else '😞 Négatif'}")

In [ ]:
# ── Distribution de la longueur des critiques ────────────────────────────────
lengths = [len(seq) for seq in train_data]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=50, color='royalblue', edgecolor='white')
ax.axvline(np.mean(lengths), color='tomato', linestyle='--', linewidth=2, label=f'Moyenne: {np.mean(lengths):.0f} mots')
ax.set_title('Distribution de la longueur des critiques (train)', fontsize=12)
ax.set_xlabel('Nombre de mots'); ax.set_ylabel("Nombre de critiques")
ax.legend()
plt.tight_layout(); plt.show()

print(f"💡 Longueur min: {min(lengths)} | max: {max(lengths)} | médiane: {np.median(lengths):.0f}")
print("   C'est pourquoi on ne peut pas directement utiliser ces séquences dans un Dense :")
print("   elles ont des longueurs VARIABLES, alors qu'un Dense attend une taille FIXE.")

### 💡 Pourquoi le One-Hot Encoding multi-label ?

Les séquences ont des **longueurs variables** — impossible de les donner directement à une couche `Dense`, qui attend une taille fixe. 

**Solution** : transformer chaque critique en un vecteur de **10 000 dimensions** (taille du vocabulaire), où chaque position `i` vaut `1` si le mot d'indice `i` apparaît dans la critique, `0` sinon.

Exemple : la séquence `[3, 5]` devient un vecteur de 10 000 zéros, sauf aux indices 3 et 5 qui valent 1.

```
[3, 5]  →  [0, 0, 0, 1, 0, 1, 0, 0, ..., 0]   (10 000 valeurs)
              ↑favorise position 3    ↑position 5
```

Cette représentation est un **sac de mots** (bag-of-words) : on perd l'ordre des mots, mais on garde l'information de présence/absence — suffisant pour ce problème de classification de sentiment.

In [ ]:
# ── Fonction de vectorisation (one-hot encoding multi-label) ─────────────────
def vectorize_sequences(sequences, dimension=10000):
    """
    Convertit une liste de séquences d'entiers en matrice binaire.

    Args:
        sequences : liste de listes d'entiers (les critiques encodées)
        dimension : taille du vocabulaire (10 000 ici)

    Returns:
        results : array numpy de forme (n_critiques, dimension)
    """
    # Créer une matrice de zéros de forme (len(sequences), dimension)
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0  # Mettre des 1 aux indices présents dans la séquence
    return results


# ── Application sur train et test ─────────────────────────────────────────────
print("🔄 Vectorisation des données (cela peut prendre quelques secondes)...")
x_train = vectorize_sequences(train_data, dimension=num_words)
x_test  = vectorize_sequences(test_data,  dimension=num_words)

# Les labels sont déjà des scalaires 0/1, on les convertit juste en float
y_train = train_labels.astype('float32')
y_test  = test_labels.astype('float32')

print(f"\n✅ Vectorisation terminée !")
print(f"   x_train : {x_train.shape}")
print(f"   x_test  : {x_test.shape}")
print(f"\n📋 Exemple — la critique [3, 5] vectorisée aurait des 1 aux indices 3 et 5 :")
print(f"   x_train[0][:10] = {x_train[0][:10]}")
print(f"   Nombre de mots uniques dans cette critique : {int(x_train[0].sum())}")

In [ ]:
# ── Découpage Train / Validation / Test ──────────────────────────────────────
# Le test set est déjà séparé par Keras. On découpe le train en train + validation.

VAL_SIZE = 10_000  # Taille classique pour ce dataset (sur 25 000 critiques train)

x_val   = x_train[:VAL_SIZE]
x_train_final = x_train[VAL_SIZE:]

y_val   = y_train[:VAL_SIZE]
y_train_final = y_train[VAL_SIZE:]

print(f"📐 Répartition finale des données :")
print(f"   Train      : {len(x_train_final):>6} critiques")
print(f"   Validation : {len(x_val):>6} critiques")
print(f"   Test       : {len(x_test):>6} critiques")

---
## 🧠 PARTIE 2 — Construction du modèle

### Architecture
Comme l'entrée est un simple vecteur (et non une image ou une séquence ordonnée), un **empilement de couches Dense avec ReLU** est tout à fait adapté :

```
Input (10 000,) → Dense(16, relu) → Dense(16, relu) → Dense(1, sigmoid)
```

- **ReLU** : introduit de la non-linéarité, permettant au réseau d'apprendre des frontières de décision complexes
- **Sigmoid en sortie** : produit une probabilité entre 0 et 1 (proba d'être une critique positive)
- **binary_crossentropy** : loss adaptée à une cible binaire avec sortie sigmoid (mesure la distance entre deux distributions de probabilité)

In [ ]:
# ── Construction du réseau feedforward ────────────────────────────────────────
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(num_words,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Sortie : probabilité (critique positive)
])

model.summary()

In [ ]:
# ── Compilation du modèle ─────────────────────────────────────────────────────
model.compile(
    optimizer='rmsprop',          # RMSprop : bon choix classique pour ce type de réseau
    loss='binary_crossentropy',   # Loss adaptée à la classification binaire + sigmoid
    metrics=['accuracy']
)

print("✅ Modèle compilé :")
print("   Optimiseur : RMSprop")
print("   Loss       : binary_crossentropy")
print("   Métrique   : accuracy")

---
## 🚀 PARTIE 3 — Entraînement du modèle

**Objectif :** Entraîner sur 20 epochs avec un batch size de 512, en surveillant la validation.

In [ ]:
# ── Entraînement initial (20 epochs) ──────────────────────────────────────────
history = model.fit(
    x_train_final, y_train_final,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

---
## 📊 PARTIE 4 — Évaluation du modèle

**Objectif :** Visualiser les courbes pour détecter l'overfitting, puis ré-entraîner avec le nombre optimal d'epochs.

In [ ]:
# ── Visualisation des courbes Loss et Accuracy ───────────────────────────────
history_dict = history.history

loss_values     = history_dict['loss']
val_loss_values = history_dict['val_loss']
acc_values      = history_dict['accuracy']
val_acc_values  = history_dict['val_accuracy']

epochs_range = range(1, len(loss_values) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Graphique Loss ────────────────────────────────────────────────────────────
axes[0].plot(epochs_range, loss_values,     'o-', color='tomato',    linewidth=2, label='Training Loss')
axes[0].plot(epochs_range, val_loss_values, 's-', color='royalblue', linewidth=2, label='Validation Loss')
axes[0].set_title('Loss — Training vs Validation', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

# ── Graphique Accuracy ────────────────────────────────────────────────────────
axes[1].plot(epochs_range, acc_values,     'o-', color='tomato',    linewidth=2, label='Training Accuracy')
axes[1].plot(epochs_range, val_acc_values, 's-', color='royalblue', linewidth=2, label='Validation Accuracy')
axes[1].set_title('Accuracy — Training vs Validation', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Historique d\'entraînement (20 epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Identification de l'epoch optimal ────────────────────────────────────────
# On cherche l'epoch où la validation loss est minimale
# (au-delà, le modèle commence à overfitter)

optimal_epoch = int(np.argmin(val_loss_values)) + 1  # +1 car epochs commencent à 1
min_val_loss  = min(val_loss_values)

print(f"🔍 Analyse des courbes :")
print(f"   Validation loss minimale : {min_val_loss:.4f} à l'epoch {optimal_epoch}")
print(f"   Validation loss finale (epoch 20) : {val_loss_values[-1]:.4f}")

if val_loss_values[-1] > min_val_loss * 1.05:
    print(f"\n⚠️  Signe d'overfitting détecté : la val_loss remonte après l'epoch {optimal_epoch}.")
    print(f"   → On va ré-entraîner avec {optimal_epoch} epochs seulement.")
else:
    print(f"\n✅ Pas de surapprentissage marqué sur cette plage d'epochs.")

### 📝 Analyse de l'overfitting

En observant les courbes ci-dessus, on remarque généralement le pattern suivant avec ce type de modèle sur IMDB :

- La **training loss** diminue de façon monotone à chaque epoch — le modèle apprend de mieux en mieux à prédire les exemples qu'il voit.
- La **validation loss** diminue d'abord, atteint un minimum vers l'epoch 3-5, puis **remonte** progressivement.
- Ce moment où les deux courbes divergent est le signal classique de l'**overfitting** : le modèle commence à mémoriser des particularités du jeu d'entraînement (mots rares, tournures spécifiques) qui ne généralisent pas aux nouvelles critiques.

**Pourquoi continuer 20 epochs n'aide pas** : au-delà du minimum de validation loss, chaque epoch supplémentaire optimise le modèle pour le bruit du train set, au détriment de sa capacité de généralisation — même si la training accuracy continue de grimper vers 99-100%.

**Solution** : ré-entraîner un nouveau modèle (poids réinitialisés) en s'arrêtant à l'epoch optimal identifié ci-dessus.

In [ ]:
# ── Ré-entraînement avec le nombre optimal d'epochs ──────────────────────────
# On reconstruit un modèle NEUF (poids réinitialisés) pour un entraînement propre

model_final = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(num_words,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_final.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"🏋️  Ré-entraînement avec {optimal_epoch} epochs (nombre optimal identifié)...\n")

# On entraîne sur l'ENSEMBLE des données d'entraînement (train + validation)
# pour maximiser les données disponibles, une fois le nombre d'epochs fixé
history_final = model_final.fit(
    x_train, y_train,
    epochs=optimal_epoch,
    batch_size=512,
    validation_data=(x_val, y_val),  # On garde un suivi de validation par cohérence
    verbose=1
)

In [ ]:
# ── Évaluation finale sur le jeu de TEST ─────────────────────────────────────
test_loss, test_accuracy = model_final.evaluate(x_test, y_test, verbose=0)

print("📊 PERFORMANCE FINALE SUR LE JEU DE TEST :\n")
print(f"   Test Loss     : {test_loss:.4f}")
print(f"   Test Accuracy : {test_accuracy*100:.2f}%")

---
## 🔍 PARTIE 5 — Analyse des résultats

**Objectif :** Comparer les métriques d'entraînement et de validation, et rapporter les résultats finaux.

In [ ]:
# ── Comparaison : entraînement original (20 epochs) vs optimal ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, loss_values,     color='tomato',  alpha=0.5, linewidth=1.5, label='Train (20 epochs)')
axes[0].plot(epochs_range, val_loss_values, color='royalblue', alpha=0.5, linewidth=1.5, label='Val (20 epochs)')
axes[0].axvline(optimal_epoch, color='green', linestyle='--', linewidth=2,
                label=f'Epoch optimal ({optimal_epoch})')
axes[0].set_title('Loss — Identification du point d\'arrêt optimal', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(fontsize=9)

metrics_comparison = ['Train\n(epoch final)', 'Val\n(epoch optimal)', 'Test\n(modèle final)']
acc_comparison = [
    history_final.history['accuracy'][-1] * 100,
    history_final.history['val_accuracy'][-1] * 100,
    test_accuracy * 100
]
colors_bar = ['tomato', 'royalblue', 'seagreen']
bars = axes[1].bar(metrics_comparison, acc_comparison, color=colors_bar, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, acc_comparison):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Accuracy — Comparaison finale', fontsize=12)
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# ── Exemples de prédictions sur le jeu de test ───────────────────────────────
predictions = model_final.predict(x_test[:10], verbose=0).flatten()

print("📋 Exemples de prédictions :\n")
print(f"{'Critique':<10} {'Vrai label':<15} {'Probabilité':<15} {'Prédiction':<12} {'Correct?'}")
print("-" * 65)
for i in range(10):
    true_label = 'Positif' if y_test[i] == 1 else 'Négatif'
    pred_label = 'Positif' if predictions[i] > 0.5 else 'Négatif'
    correct = '✅' if pred_label == true_label else '❌'
    print(f"#{i:<9} {true_label:<15} {predictions[i]:.4f}{'':<8} {pred_label:<12} {correct}")

In [ ]:
# ── Matrice de confusion sur le test set complet ─────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

all_predictions = model_final.predict(x_test, verbose=0).flatten()
all_pred_labels = (all_predictions > 0.5).astype(int)

cm = confusion_matrix(y_test, all_pred_labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Négatif', 'Positif'], yticklabels=['Négatif', 'Positif'], ax=ax)
ax.set_title('Matrice de Confusion — Test Set', fontsize=12)
ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
plt.tight_layout(); plt.show()

print("\n📋 Rapport de classification :")
print(classification_report(y_test, all_pred_labels, target_names=['Négatif', 'Positif'], digits=3))

### 📝 Analyse comparative Train / Validation / Test

**Comportement du modèle** : en comparant l'accuracy d'entraînement, de validation et de test, on observe typiquement que :
- L'**accuracy d'entraînement** (mesurée sur les données vues pendant l'apprentissage) est la plus élevée, car le modèle a directement optimisé ses poids sur ces exemples.
- L'**accuracy de validation** est légèrement inférieure mais reste proche, ce qui confirme que l'epoch optimal a été correctement identifié — le modèle généralise raisonnablement bien sans sur-apprentissage excessif.
- L'**accuracy de test** (sur des données jamais vues, ni en entraînement ni en validation) est l'estimation la plus fiable de la performance réelle du modèle en conditions réelles, et devrait être très proche de l'accuracy de validation si le nombre d'epochs a été bien calibré.

**Résultat final attendu** : avec cette architecture simple (2 couches Dense de 16 neurones), on obtient typiquement une accuracy de test autour de **86-88%** sur IMDB — un excellent résultat pour un modèle aussi simple, qui démontre la puissance de la représentation bag-of-words pour ce type de tâche de sentiment analysis.

**Limites de cette approche** : le one-hot encoding ignore complètement l'**ordre des mots** — "this movie is not good" et "this movie is good, not bad" auraient des représentations très similaires malgré des sentiments potentiellement différents. Des architectures plus avancées (RNN, LSTM, Transformers) capturent cette information séquentielle et peuvent améliorer encore les performances.

---
## 🎓 Conclusion

Vous avez construit un pipeline complet de classification de texte binaire :

| Partie | Ce qu'on a fait |
|--------|----------------|
| **1 - Prétraitement** | Chargé IMDB, vectorisé les séquences (one-hot 10 000 dim), split train/val/test |
| **2 - Modèle** | Réseau Dense(16, relu) → Dense(16, relu) → Dense(1, sigmoid), compilé avec RMSprop |
| **3 - Entraînement** | Entraîné sur 20 epochs, batch_size=512, avec suivi de validation |
| **4 - Évaluation** | Identifié l'epoch optimal via la validation loss minimale, ré-entraîné, évalué sur le test |
| **5 - Analyse** | Comparé train/val/test, matrice de confusion, rapport de classification |

### 💡 Points clés à retenir
- Le **one-hot encoding multi-label** transforme des séquences de longueur variable en vecteurs de taille fixe
- Pour une cible binaire avec sortie sigmoid, la **binary_crossentropy** est la loss appropriée
- Surveiller la **validation loss** (et non l'accuracy) permet de détecter précisément le début de l'overfitting
- Ré-entraîner avec le nombre d'epochs optimal est une pratique simple mais très efficace contre le surapprentissage

### 🚀 Pour aller plus loin :
- Essayer un encodage **TF-IDF** au lieu du simple one-hot binaire
- Utiliser des **embeddings de mots** (Word2Vec, GloVe) pour capturer la sémantique
- Passer à des architectures séquentielles (**LSTM, GRU**) ou des **Transformers** pour exploiter l'ordre des mots